### **Setup: Imports und Initialisierung**

In [1]:
import sys
import os
import pandas as pd

sys.path.append(os.path.abspath('..'))

from quinex_application.mongos_store import MongoStore
from quinex_application.quinex import Quinex
from quinex_application.visualisation_quinex import VisualisationQuinex
from quinex_application.oeo_data import OEOData
from quinex_application.levenshtein import LevenshteinDistance
from quinex_application.techdb import TechDB

In [2]:
quinex = Quinex()
mongodb = MongoStore(quinex, database_name="quinex_db_query4", collection_name="papers_2025")
visualisation = VisualisationQuinex()
oeo = OEOData("../data/oeo/etd.xlsx", "../data/oeo/oeo-full.owl", "../data/oeo/oeo.xlsx")
levenshtein = LevenshteinDistance()
tb = TechDB()

### **Load Analysis into DataFrame**

In [3]:
analysis = mongodb.get_collection("papers_2024")
df_path = "../data/quinex/df_quinex.parquet"

df_quinex = visualisation.create_df(analysis)
# visualisation.save_df(df_quinex, df_path)

display(df_quinex.head(5))
print(len(df_quinex))

""


0


In [ ]:
import pandas as pd
df_quinex = pd.read_parquet("../data/quinex/df_quinex.parquet")

### **Metadata Results**

In [ ]:
# Alle Paper-IDs, die in analysis vorkommen (mit Analyse)
analysis_ids = {paper["_id"] for paper in analysis if paper.get("analysis_result")}

# Alle Paper-IDs, die tatsächlich im DataFrame gelandet sind
df_ids = set(df_quinex["paper_id"].unique())

# Fehlende Paper mit Analyse, aber ohne quantitative Aussage
missing = analysis_ids - df_ids

print(f"🧾 Total analyzed papers (with analysis_result): {len(analysis_ids)}")
print(f"✅ Papers in DataFrame: {len(df_ids)}")
print(f"❌ Papers with no valid quantitative statements: {len(missing)}")

# Optional: welche das sind
print("Missing paper_ids:", missing)

In [ ]:
# Gesamtanzahl der quantitativen Aussagen
total_statements = len(df_quinex)

# Durchschnittliche Anzahl an Aussagen pro Paper
avg_statements_per_paper = total_statements / len(df_ids)

print(f"📊 Total quantitative statements: {total_statements}")
print(f"➗ Average statements per paper: {avg_statements_per_paper:.2f}")

### **Apply Levenshtein Method with TechDB Hirarchy**

In [ ]:
en_list = tb.save_entities()
prop_list = tb.save_properties()

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein.parquet"
# df_quinex = visualisation.load_df(df_path)

df_quinex[['entity_mapping', 'entity_match_score']] = df_quinex['entity'].apply(lambda x: pd.Series(levenshtein.get_best_match(x, en_list, "similarity")))
#visualisation.save_df(df_quinex, df_path)

In [ ]:
df_quinex[['property_mapping', 'property_match_score']] = df_quinex['property'].apply(lambda x: pd.Series(levenshtein.get_best_match(x, prop_list, "similarity")))

In [12]:
df_levenshtein = df_quinex[(df_quinex['entity_match_score'] >= 0.75) & (df_quinex['property_match_score'] >=0.75)]
display(df_levenshtein.head(10))
print(len(df_levenshtein))

,paper_id,entity,property,quantity,unit,quote,temporal_scope_text,spatial_scope_text,entity_mapping,entity_match_score,property_mapping,property_match_score
18,10.1016/j.rser.2025.115648,carbon capture and storage,investment costs,~ 2 %,"[{'uri': 'http://qudt.org/vocab/unit/PERCENT', 'exponent': 1, 'year': None}]","This analysis does not include carbon capture and storage operations, which would marginally impact the investment costs by around 2 % [18].",,,carbon capture and storage,1.000000,investment cost,0.937500
221,10.1016/j.trpro.2025.02.025,electric motor,power,= 500kW,"[{'uri': 'http://qudt.org/vocab/unit/KiloW', 'exponent': 1, 'year': None}]","5 knots, power of the electric motor is just under 500kW, and propeller speed demand is 350 rpm.",,,electric motor,1.000000,power,1.000000
275,10.1080/15567036.2021.1922551,Biomass,potential,= 144 million tonnes per year,"[{'uri': 'http://qudt.org/vocab/unit/TONNE', 'exponent': 1, 'year': None}, {'uri': 'http://qudt.org/vocab/unit/YR', 'exponent': -1, 'year': None}]","Biomass potential in Nigeria is estimated at 144 million tonnes per year and consists of fuel-wood, agricultural waste, animal dung/poultry droppings and municipal solid waste (MSW) [7] as shown in Figure 3 .",,Unguwar Dogo (7.4023°N 11.61172°E) located in Bakogori Local Government Area of Katsina state Nigeria,biomass,0.857143,potential,1.000000
764,10.1016/j.rser.2024.114944,freight transport,energy consumption,= 199 PJ,"[{'uri': 'http://qudt.org/vocab/unit/PetaJ', 'exponent': 1, 'year': None}]","In addition, the energy consumption of freight transport in SUFF is 199 PJ lower than in TECH due to a shift from road and air to rail and water and a decrease in the volume of goods transported.",,,freight transport,1.000000,energy consumption value,0.750000
811,10.1016/j.apenergy.2024.124687,biogas,volume,~ 18 Mtoe,"[{'uri': 'http://qudt.org/vocab/unit/MegaTOE', 'exponent': 1, 'year': None}]","Literature review and scope\nAs a continent, Europe is currently the largest producer of biogas (around 18 Mtoe in 2018), mainly derived from crops and animal manure, with Germany being by far the largest market, followed by Denmark, France, Italy and the Netherlands [7].",in 2018,Europe,biogas,1.000000,volume,1.000000
937,10.1016/j.applthermaleng.2024.124513,ships,energy consumption,zero or low-emission,[],"This allows for the determination of onboard solutions to reduce energy consumptions and improve safety, as well as to comply with future regulations [32]. From this perspective, the influence of different imposed policies (e.g. carbon pricing) and the related optimal energy system can be evaluated dynamically by using modelling approaches [33] that allow assessing whether the investigated energy systems adhere to imposed regulations and assist governments and lawmakers in proposing new limits [34]. In this context, dynamic analysis for ships is crucial to evaluate their dynamic loads, also when considering ports as energy communities with various users [35], including ships, since it is fundamental for integrating cold ironing to enhance energy efficiency and reduce emissions [36].\n\nThese objectives can be accomplished through numerical simulations and the integration of various models targeting specific aspects of vessel design or operation [37]. These models can be seamlessly incorporated into digital twin platforms, applicable throughout a vessel lifecycle—from initial design to operation, retrofitting, and end of life [38]. However, despite efforts to integrate complex multi-physics simulations into a unified digital twin platform, a general lack of data undermines the reliability of these tools [39]. By mapping and utilizing available and relevant data, it is possible to effectively employ integrated models of onboard energy and power systems for designing and operating zero or low-emission vessels and for sustainability upgrades throughout their lifecycle [40].",,,ship,0.800000,energy consumption value,0.750000
1095,

279


In [11]:
visualisation.save_df(df_levenshtein, "../data/quinex/df_quinex_prop_ent_mapped.parquet")

✅ New DataFrame saved to '../data/quinex/df_quinex_prop_ent_mapped.parquet'


In [ ]:
print(df_levenshtein["entity_mapping"].nunique())
print(df_levenshtein["entity_mapping"].unique())

### **Visualise Results**

In [ ]:
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=25, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_values(df_quinex, column="entity", top_n=25, match_column="best_oeo_match", score_column="match_score", figsize=(6,6))

In [ ]:
fig = visualisation.plot_top_properties_of_entity(df_quinex, "electrolyzer", top_n=10, ignore_empty=True, figsize=(6,6))
fig = visualisation.plot_top_quantity_of_property(df_quinex, "electrolyzer", "rated power", top_n=8, ignore_empty=True, figsize=(6,6))

In [ ]:
df_path = "../data/quinex/df_quinex_levenshtein.parquet"
fig = visualisation.plot_levenshtein_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])
fig = visualisation.plot_unique_entities_over_score_limits(df_path, score_limits=[0.7, 0.75, 0.8, 0.85, 0.9])